# Retailrocket Dataset Download

This notebook is designed to run in Google Colab and uses `data.py` to download a Kaggle dataset.

In [1]:
import sys
import csv
from pathlib import Path

project_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from data import download_kaggle_dataset, preprocess_retailrocket_data

Download the Retailrocket dataset into the local `data/` folder so the preprocessing step can work with the raw CSV files.

In [2]:
dataset_name = "retailrocket/ecommerce-dataset"
output_path = download_kaggle_dataset(dataset_name, output_dir=project_root / "data")
print(f"Downloaded dataset into: {output_path}")
sorted(str(path.relative_to(output_path)) for path in output_path.rglob("*") if path.is_file())[:20]

/home/mrjoezhong/miniconda3/envs/serko/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Downloaded dataset into: /home/mrjoezhong/serko-ds-test/data


['.complete/datasets/retailrocket/ecommerce-dataset/2/bundle.complete',
 'cate_properties.csv',
 'category_lookup.csv',
 'category_tree.csv',
 'events.csv',
 'events_merged.csv',
 'events_time_range.csv',
 'item_properties.csv',
 'item_properties_part1.csv',
 'item_properties_part2.csv',
 'non_numeric_properties.csv',
 'numeric_properties.csv',
 'property_id_map.csv',
 'property_mean_std.csv']

Run the shared preprocessing helper once to prepare the derived Retailrocket tables.

`preprocess_retailrocket_data(...)` currently does four things:
- builds `category_lookup.csv` from `category_tree.csv`
- computes the category leaf-depth summary returned as `leaf_depth_statistics`
- merges `item_properties_part1.csv` and `item_properties_part2.csv` into `item_properties.csv`
- splits the merged item properties into `cate_properties.csv` for `categoryid` and `available`, and `non_numeric_properties.csv` for the remaining non-numeric properties

In [3]:
preprocess_outputs = preprocess_retailrocket_data(output_path)
category_lookup_path = preprocess_outputs["category_lookup_path"]
leaf_depth_stats = preprocess_outputs["leaf_depth_statistics"]
merged_item_properties_path = preprocess_outputs["merged_item_properties_path"]

print(f"Generated category lookup table at: {category_lookup_path}")
print(f"Merged item properties file at: {merged_item_properties_path}")
print("Leaf depth statistics:")
for depth, count in leaf_depth_stats.items():
    print(f"depth={depth}, count={count}")

with category_lookup_path.open(newline="", encoding="utf-8") as handle:
    preview_rows = list(csv.reader(handle))[:10]

preview_rows

Scanning event timestamps: 2756101row [00:05, 492751.92row/s]


Finished generate_events_time_range -> /home/mrjoezhong/serko-ds-test/data/events_time_range.csv


Scanning event timestamps: 2756101row [00:05, 493410.75row/s]
Grouping events by visitor-item: 2756101row [00:08, 340669.27row/s]
Writing merged visitor-item events: 100%|██████████| 2145179/2145179 [00:13<00:00, 159410.50row/s]


Finished merge_events_by_visitor_item -> /home/mrjoezhong/serko-ds-test/data/events_merged.csv (negative=1,931,118, positive=214,061, negative_cold_start=102, positive_cold_start=13,527)
Finished generate_category_lookup_table -> /home/mrjoezhong/serko-ds-test/data/category_lookup.csv
Finished merge_item_properties_files -> /home/mrjoezhong/serko-ds-test/data/item_properties.csv


Loading item property rows: 20275902row [01:29, 227276.52row/s]
Deduplicating item-property groups: 100%|██████████| 10368178/10368178 [00:20<00:00, 504325.98group/s]


Finished load_item_property_row_groups -> numeric=1,201,114, category=1,031,473, non_numeric=10,700,916
Finished process_numeric_property_values -> /home/mrjoezhong/serko-ds-test/data/numeric_properties.csv
Finished process_category_property_values -> /home/mrjoezhong/serko-ds-test/data/cate_properties.csv
Finished process_non_numeric_property_values -> /home/mrjoezhong/serko-ds-test/data/non_numeric_properties.csv
Generated category lookup table at: /home/mrjoezhong/serko-ds-test/data/category_lookup.csv
Merged item properties file at: /home/mrjoezhong/serko-ds-test/data/item_properties.csv
Leaf depth statistics:
depth=1, count=1
depth=2, count=50
depth=3, count=525
depth=4, count=632
depth=5, count=86
depth=6, count=13


[['level_1_category_id',
  'level_2_category_id',
  'level_3_category_id',
  'level_4_category_id',
  'level_5_category_id',
  'level_6_category_id'],
 ['140', '61', '323', '1558', '', ''],
 ['140', '61', '323', '', '', ''],
 ['140', '61', '897', '120', '', ''],
 ['140', '61', '897', '1098', '', ''],
 ['140', '61', '897', '1317', '', ''],
 ['140', '61', '897', '1540', '', ''],
 ['140', '61', '897', '1588', '', ''],
 ['140', '61', '897', '', '', ''],
 ['140', '61', '1003', '', '', '']]

Build or load the saved all-item embedding table produced by `model.get_all_items_embedding(...)`.

In [ ]:
from model import get_all_items_embedding

all_item_embeddings = get_all_items_embedding(dataset_path=output_path)
print(f'all_item_embeddings shape: {tuple(all_item_embeddings.shape)}')
print(f'all_item_embeddings device: {all_item_embeddings.device}')
all_item_embeddings[:2, :8]
